In [ ]:
import torch
import torch.nn as nn
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import timm
from torchvision import transforms

# Define the Enhanced ViT model (with a small modification for GradCAM)
class EnhancedViT(nn.Module):
    def __init__(self, dropout_rate=0.3, hidden_dim=512):
        super(EnhancedViT, self).__init__()
        # Load a pre-trained ViT backbone and remove its classification head.
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=True)
        self.vit.head = nn.Identity()

        if hasattr(self.vit, 'embed_dim'):
            self.embed_dim = self.vit.embed_dim
        else:
            self.embed_dim = self.vit.head.in_features

        # An attention mechanism to pool the patch tokens.
        self.attention_layer = nn.Sequential(
            nn.Linear(self.embed_dim, 1)
        )

        # Custom classifier head.
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.embed_dim),
            nn.Dropout(dropout_rate),
            nn.Linear(self.embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, 2)  # Binary classification (2 classes)
        )

    def forward(self, x):
        # Get patch token embeddings; shape: [B, num_tokens, embed_dim]
        tokens = self.vit.forward_features(x)
        # Register a hook on the tokens to capture gradients during backprop.
        tokens.register_hook(self.save_tokens_grad)
        self.tokens = tokens  # Save activations for later use in GradCAM.

        # Compute attention scores and aggregate tokens.
        attn_scores = self.attention_layer(tokens)
        attn_weights = torch.softmax(attn_scores, dim=1)
        weighted_feature = torch.sum(attn_weights * tokens, dim=1)
        out = self.classifier(weighted_feature)
        return out

    def save_tokens_grad(self, grad):
        self.tokens_grad = grad

# GradCAM computation function.
def compute_gradcam(model, input_tensor, target_class=None):
    model.eval()
    output = model(input_tensor)
    if target_class is None:
        target_class = output.argmax(dim=1).item()

    # Zero all existing gradients.
    model.zero_grad()
    # Compute the score for the target class.
    score = output[0, target_class]
    score.backward()

    # Retrieve the patch token activations and gradients.
    # (Assuming the first token is the [CLS] token and should be ignored.)
    tokens = model.tokens[:, 1:, :]        # shape: [1, num_tokens-1, embed_dim]
    tokens_grad = model.tokens_grad[:, 1:, :]  # same shape

    # For a 224x224 image with patch size 16, there are 14x14 patches.
    batch_size, num_tokens, embed_dim = tokens.shape
    h = w = int(np.sqrt(num_tokens))  # Should be 14 for ViT-base.

    # Reshape tokens and gradients into a spatial map.
    tokens_reshaped = tokens.reshape(batch_size, h, w, embed_dim).permute(0, 3, 1, 2)
    tokens_grad_reshaped = tokens_grad.reshape(batch_size, h, w, embed_dim).permute(0, 3, 1, 2)

    # Compute channel-wise weights by global-average-pooling the gradients.
    weights = tokens_grad_reshaped.mean(dim=(2, 3), keepdim=True)  # shape: [1, embed_dim, 1, 1]

    # Combine weights with the activation maps.
    cam = (weights * tokens_reshaped).sum(dim=1, keepdim=True)  # shape: [1, 1, h, w]
    cam = torch.relu(cam)
    cam = cam.squeeze().cpu().detach().numpy()

    # Normalize the CAM to [0, 1] and resize to the input image size (224, 224).
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    cam = cv2.resize(cam, (224, 224))

    # Get the confidence score for the predicted class.
    confidence = torch.softmax(output, dim=1)[0, target_class].item()

    return cam, target_class, confidence

# Preprocess a single image.
def preprocess_image(image_path):
    transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
    ])
    image = Image.open(image_path).convert("RGB")
    original_image = np.array(image.resize((224,224)))
    image_tensor = transform(image).unsqueeze(0)
    return image_tensor, original_image

# Visualize the GradCAM result for one image.
def visualize_gradcam(model, image_path, class_names):
    input_tensor, original_image = preprocess_image(image_path)
    cam, pred_class, confidence = compute_gradcam(model, input_tensor)

    # Create a heatmap from the CAM.
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(original_image, 0.5, heatmap, 0.5, 0)

    # Plot the original image, CAM map, and overlay.
    plt.figure(figsize=(15,5))

    plt.subplot(1,3,1)
    plt.imshow(original_image)
    plt.title("Original Image")
    plt.axis('off')

    plt.subplot(1,3,2)
    plt.imshow(cam, cmap='jet')
    plt.title("GradCAM Map")
    plt.axis('off')

    plt.subplot(1,3,3)
    plt.imshow(overlay)
    plt.title(f"Overlay\nPredicted: {class_names[pred_class]}, Conf: {confidence:.2%}")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    # Initialize the model.
    model = EnhancedViT()
    # Optionally, load your trained weights:
    model.load_state_dict(torch.load('/content/drive/My Drive/FYP/VIT2_HQ2_20241224/final_model_vit_20241224.pth', map_location='cpu'))
    class_names = ['Glaucous_Winged_Gull', 'Slaty_Backed_Gull']

    # Set the path to a single test image.
    image_path = '/content/097.jpg'
    visualize_gradcam(model, image_path, class_names)


<ipython-input-10-fb5d750ecef9>:145: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('/content/drive/My Drive/FYP/VIT2_HQ2_20241224/final_mode

RuntimeError: Error(s) in loading state_dict for EnhancedViT:
	Missing key(s) in state_dict: "attention_layer.0.weight", "attention_layer.0.bias", "classifier.0.weight", "classifier.0.bias", "classifier.2.weight", "classifier.2.bias", "classifier.5.weight", "classifier.5.bias". 
	Unexpected key(s) in state_dict: "vit.head.weight", "vit.head.bias". 

In [ ]:
!pip install captum

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 889.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 79.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [ ]:
!pip install lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=e46f050ce4e6e0b067294490e1377e01d0ca37882b346501514ec7034830edeb
  Stored in directory: /root/.cache/pip/wheels/85/fa/a3/9c2d44c9f3cd77cf4e533b58900b2bf4487f2a17e8ec212a3d
Successfully built lime


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import torch.nn as nn
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import timm
import os
from typing import Tuple, Optional
from torchvision import transforms
from captum.attr import DeepLift
from datetime import datetime

# Enhanced ViT Model with Dropout
class EnhancedViT(nn.Module):
    def __init__(self, dropout_rate=0.3, hidden_dim=512):
        """
        Initializes the enhanced ViT model.
        - Loads a pre-trained ViT backbone.
        - Removes the original classification head.
        - Adds an attention mechanism to pool patch tokens.
        - Adds a custom MLP classifier head.
        """
        super(EnhancedViT, self).__init__()
        # Load a pre-trained ViT model from timm
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=True)
        # Remove the original classification head
        self.vit.head = nn.Identity()

        # Get the embedding dimension (most timm ViT models have 'embed_dim')
        if hasattr(self.vit, 'embed_dim'):
            self.embed_dim = self.vit.embed_dim
        else:
            # Fallback: use the in_features of the original head if available
            self.embed_dim = self.vit.head.in_features

        # Attention mechanism: compute an attention score for each token (patch)
        self.attention_layer = nn.Sequential(
            nn.Linear(self.embed_dim, 1)  # Outputs a scalar score per token
        )

        # Custom classifier head: LayerNorm -> Dropout -> Linear -> ReLU -> Dropout -> Linear
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.embed_dim),
            nn.Dropout(dropout_rate),
            nn.Linear(self.embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, 2)  # Binary classification (2 classes)
        )

    def forward(self, x):
        """
        Forward pass:
          1. Extract patch token embeddings via ViT's forward_features.
          2. Compute attention scores for each token.
          3. Aggregate tokens via a weighted sum.
          4. Classify the aggregated feature vector.
        """
        # Get patch token embeddings; expected shape: [batch, num_tokens, embed_dim]
        tokens = self.vit.forward_features(x)
        # Compute attention scores for each token; shape: [batch, num_tokens, 1]
        attn_scores = self.attention_layer(tokens)
        # Normalize attention scores using softmax along the token dimension
        attn_weights = torch.softmax(attn_scores, dim=1)
        # Compute the weighted sum of token embeddings to form a global feature vector
        weighted_feature = torch.sum(attn_weights * tokens, dim=1)  # Shape: [batch, embed_dim]
        # Pass the aggregated features through the classifier head
        out = self.classifier(weighted_feature)
        return out


class DeepLIFTAnalyzer:
    def __init__(self, model: nn.Module, class_names: list,
                 save_dir: str = None,
                 device: str = 'cuda' if torch.cuda.is_available() else 'cpu'):
        self.model = model.to(device)
        self.device = device
        self.class_names = class_names
        self.save_dir = save_dir
        if save_dir:
            os.makedirs(save_dir, exist_ok=True)

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])
        self.deep_lift = DeepLift(self.model)

class DeepLIFTAnalyzer:
    def __init__(self, model: nn.Module, class_names: list,
                 save_dir: str = None,
                 device: str = 'cuda' if torch.cuda.is_available() else 'cpu'):
        self.model = model.to(device)
        self.device = device
        self.class_names = class_names
        self.save_dir = save_dir
        if save_dir:
            os.makedirs(save_dir, exist_ok=True)

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])
        self.deep_lift = DeepLift(self.model)

    def preprocess_image(self, image_path: str) -> Tuple[torch.Tensor, np.ndarray]:
        """Preprocess image and return both tensor and original image."""
        original_image = Image.open(image_path).convert("RGB")
        original_image = original_image.resize((224, 224))
        original_np = np.array(original_image)

        image_tensor = self.transform(original_image).unsqueeze(0).to(self.device)
        return image_tensor, original_np

    def generate_deeplift_attribution(self, image_tensor: torch.Tensor) -> Tuple[np.ndarray, int, float]:
        """Generate DeepLIFT attribution map with confidence score."""
        self.model.eval()

        baseline = torch.zeros_like(image_tensor).to(self.device)

        with torch.no_grad():
            output = self.model(image_tensor)
            probabilities = torch.nn.functional.softmax(output, dim=1)
            confidence_score = torch.max(probabilities).item()
            pred_class = output.argmax().item()

        try:
            attribution = self.deep_lift.attribute(
                image_tensor,
                baselines=baseline,
                target=pred_class,
                return_convergence_delta=False
            )

            attribution = attribution.detach().cpu().numpy()
            attribution = np.mean(np.abs(attribution), axis=1)
            attribution = attribution.squeeze()
            attribution = cv2.resize(attribution, (224, 224))
            attribution = (attribution - attribution.min()) / (attribution.max() - attribution.min() + 1e-8)
            attribution = cv2.GaussianBlur(attribution, (3, 3), 0)

        except Exception as e:
            print(f"Error generating attribution: {str(e)}")
            attribution = np.ones((224, 224)) * 0.5

        return attribution, pred_class, confidence_score

    def create_overlay(self, original_image: np.ndarray, attribution: np.ndarray,
                      alpha: float = 0.7, colormap: int = cv2.COLORMAP_JET) -> np.ndarray:
        """Create colored overlay of attribution map on original image."""
        # Enhance the attribution map contrast
        attribution_map = np.uint8(attribution * 255)

        # # Apply contrast enhancement
        # clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
        # attribution_map = clahe.apply(attribution_map)

        # Create more vibrant heatmap
        heatmap = cv2.applyColorMap(attribution_map, colormap)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

        # USE THIS LINE TO Increase heatmap intensity
        # heatmap = cv2.addWeighted(heatmap, 1.2, np.zeros_like(heatmap), 0, 0)

        # Create overlay with adjusted alpha for better visibility
        overlay = cv2.addWeighted(original_image, alpha, heatmap, 1.0 - alpha, 0)
        return overlay

    def visualize_attribution(self, image_path: str,
                            save_path: Optional[str] = None,
                            figure_size: Tuple[int, int] = (15, 5)):
        """Visualize original image, DeepLIFT attribution, and overlay."""
        image_tensor, original_image = self.preprocess_image(image_path)
        attribution, pred_class, confidence = self.generate_deeplift_attribution(image_tensor)
        overlay_image = self.create_overlay(original_image, attribution)

        # Create figure with larger size for better visibility
        fig, axes = plt.subplots(1, 3, figsize=figure_size)

        # Original image
        axes[0].imshow(original_image)
        axes[0].set_title(f"Original\nPredicted: {self.class_names[pred_class]}\nConfidence: {confidence:.2%}",
                         fontsize=12, pad=10)
        axes[0].axis('off')

        # Attribution map with enhanced colormap
        axes[1].imshow(attribution, cmap='hot')
        axes[1].set_title('DeepLIFT Attribution Map', fontsize=12, pad=10)
        axes[1].axis('off')

        # Enhanced overlay
        axes[2].imshow(overlay_image)
        axes[2].set_title('DeepLIFT Overlay', fontsize=12, pad=10)
        axes[2].axis('off')

        plt.tight_layout()

        # Save if requested
        if save_path or self.save_dir:
            if self.save_dir:
                class_folder = os.path.join(self.save_dir, self.class_names[pred_class])
                os.makedirs(class_folder, exist_ok=True)

                base_name = os.path.splitext(os.path.basename(image_path))[0]
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                save_path = os.path.join(class_folder, f"{base_name}_deeplift_{timestamp}.png")

            plt.savefig(save_path, bbox_inches='tight', dpi=300)

        # Always display the plot
        plt.show()
        plt.close()

def analyze_dataset(analyzer: DeepLIFTAnalyzer, test_folder: str, max_images: int = None):
    """Process images in test folder up to max_images per class."""
    # Get all class folders
    class_folders = [d for d in os.listdir(test_folder)
                    if os.path.isdir(os.path.join(test_folder, d))]

    for class_folder in class_folders:
        print(f"\nProcessing class: {class_folder}")
        class_path = os.path.join(test_folder, class_folder)

        # Get all images in the class folder
        images = [f for f in os.listdir(class_path)
                 if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        # Limit number of images if specified
        if max_images:
            images = images[:max_images]

        for image in images:
            image_path = os.path.join(class_path, image)
            print(f"Processing: {image_path}")
            analyzer.visualize_attribution(image_path)
            print(f"Visualization complete for: {image}")

if __name__ == "__main__":
    # Initialize model and analyzer

    # one with more images in test dataset
    model = EnhancedViT()

    model.load_state_dict(torch.load('/content/drive/My Drive/FYP/VIT2_HQ3_20250208/final_model_vit_20250208.pth',
                                   map_location='cpu'))
    class_names = ['Glaucous_Winged_Gull', 'Slaty_Backed_Gull']

    # Set up save directory in Google Drive
    save_dir = '/content/drive/My Drive/FYP/DeepLIFT_Results'

    analyzer = DeepLIFTAnalyzer(model, class_names, save_dir=save_dir)

    # Analyze test dataset with maximum 5 images per class
    test_folder = '/content/drive/My Drive/FYP/Dataset/HQ3/train'
    analyze_dataset(analyzer, test_folder, max_images=15)

In [ ]:
import os
import warnings
from datetime import datetime
from functools import partial
from typing import Tuple, Optional

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torchvision import transforms

import timm

from captum.attr import DeepLift
import shap
from lime import lime_image
from skimage.segmentation import mark_boundaries, slic


# ================================
# 1. Define the Enhanced ViT Model
# ================================
class EnhancedViT(nn.Module):
    def __init__(self, dropout_rate: float = 0.3, hidden_dim: int = 512):
        super(EnhancedViT, self).__init__()
        # Create a pretrained ViT and remove its head
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=True)
        self.vit.head = nn.Identity()
        self.embed_dim = self.vit.embed_dim

        # A simple attention layer that produces a weight for each token
        self.attention_layer = nn.Sequential(nn.Linear(self.embed_dim, 1))

        # Final classifier
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.embed_dim),
            nn.Dropout(dropout_rate),
            nn.Linear(self.embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, 2)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        tokens = self.vit.forward_features(x)
        attn_scores = self.attention_layer(tokens)
        attn_weights = torch.softmax(attn_scores, dim=1)
        weighted_feature = torch.sum(attn_weights * tokens, dim=1)
        return self.classifier(weighted_feature)

    def get_attention_weights(self, x: torch.Tensor) -> np.ndarray:
        """Extract attention weights from the model for visualization."""
        tokens = self.vit.forward_features(x)
        attn_scores = self.attention_layer(tokens)
        attn_weights = torch.softmax(attn_scores, dim=1)
        # Return as a numpy array – note: for ViT-base the spatial map is 14x14.
        return attn_weights.detach().cpu().numpy().squeeze()


# =======================================
# 2. Define the XAI Analyzer Class
# =======================================
class XAIAnalyzer:
    def __init__(self, model: nn.Module, class_names: list,
                 save_dir: Optional[str] = None,
                 device: str = 'cuda' if torch.cuda.is_available() else 'cpu'):
        self.model = model.to(device).eval()
        self.device = device
        self.class_names = class_names
        self.save_dir = save_dir
        if self.save_dir:
            os.makedirs(self.save_dir, exist_ok=True)

        # Standard preprocessing (resize, tensor conversion, normalization)
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])

    def _get_prediction(self, image_tensor: torch.Tensor) -> Tuple[int, float]:
        """Return the predicted class index and its confidence."""
        with torch.no_grad():
            output = self.model(image_tensor)
            proba = torch.softmax(output, dim=1)[0]
            conf, pred = torch.max(proba, 0)
            return pred.item(), conf.item()

    def generate_deeplift_attribution(self, image_tensor: torch.Tensor, target: int) -> np.ndarray:
        """
        Compute DeepLIFT attributions for the given target class.
        The target must be provided since the model outputs multiple scores.
        """
        dl = DeepLift(self.model)
        baseline = torch.zeros_like(image_tensor).to(self.device)
        # Specify the target class index
        attribution = dl.attribute(image_tensor, baselines=baseline, target=target)
        return self._process_attribution(attribution)

    def generate_shap_attribution(self, image_tensor: torch.Tensor, target: int,
                                  original_image: Optional[np.ndarray] = None) -> np.ndarray:
        """
        Compute SHAP attributions for the given target class.
        The `original_image` (as a NumPy array in RGB, shape (224,224,3)) is expected in pixel space.
        """
        # If an original image is provided, use it (scaled to [0, 1]); otherwise, try to recover it.
        if original_image is not None:
            shap_input = (original_image / 255.0).astype(np.float32)
        else:
            # WARNING: if original_image is not provided then the attribution may be off.
            # You can implement an "unnormalize" function if needed.
            shap_input = np.transpose(image_tensor[0].detach().cpu().numpy(), (1, 2, 0))

        # Define a wrapper that converts the numpy input (in [0,1], shape (batch, H, W, C))
        # into a normalized torch tensor and returns only the target output.
        def shap_wrapper(x):
            if len(x.shape) == 3:
                x = np.expand_dims(x, axis=0)
            # Convert to torch tensor with channel-first ordering
            x_tensor = torch.from_numpy(x).permute(0, 3, 1, 2).float().to(self.device)
            # Apply the same normalization as in training
            mean = torch.tensor([0.485, 0.456, 0.406]).to(self.device).view(1, 3, 1, 1)
            std = torch.tensor([0.229, 0.224, 0.225]).to(self.device).view(1, 3, 1, 1)
            x_tensor = (x_tensor - mean) / std
            with torch.no_grad():
                # Return only the score for the target class
                output = self.model(x_tensor)[:, target]
                return output.detach().cpu().numpy()

        # Define a masker for images. Here we specify the expected image shape in (H,W,C) format.
        masker = shap.maskers.Partition((224, 224, 3), clustering="correlation")
        explainer = shap.PartitionExplainer(shap_wrapper, masker=masker, algorithm="auto")
        # Compute SHAP values; nsamples can be increased for better estimates.
        shap_values = explainer.shap_values(shap_input, nsamples=200, fixed_background=True)

        # The output may be a list (one array per class); here we extract the target attribution.
        if isinstance(shap_values, list):
            # Average absolute values (optionally you could choose positive or signed values)
            attribution = np.mean(np.abs(shap_values[0]), axis=0)
        else:
            attribution = np.mean(np.abs(shap_values), axis=0)

        # Resize and normalize the attribution map
        attribution = cv2.resize(attribution, (224, 224))
        attribution = (attribution - attribution.min()) / (attribution.max() - attribution.min() + 1e-8)
        return attribution

    def generate_lime_explanation(self, image_path: str) -> Tuple[np.ndarray, np.ndarray]:
        """
        Generate a LIME explanation.
        Uses skimage's slic for segmentation.
        """
        explainer = lime_image.LimeImageExplainer()
        image = np.array(Image.open(image_path).convert("RGB").resize((224, 224))) / 255.0

        def batch_predict(images):
            # Convert a list/array of images (assumed to be in range [0,1] and shape (batch, H, W, C))
            # into a torch tensor in normalized form.
            tensor = torch.from_numpy(np.array(images)).permute(0, 3, 1, 2).float().to(self.device)
            # Apply normalization (same as training)
            mean = torch.tensor([0.485, 0.456, 0.406]).to(self.device).view(1, 3, 1, 1)
            std = torch.tensor([0.229, 0.224, 0.225]).to(self.device).view(1, 3, 1, 1)
            tensor = (tensor - mean) / std
            with torch.no_grad():
                outputs = self.model(tensor)
                return torch.softmax(outputs, dim=1).cpu().numpy()

        explanation = explainer.explain_instance(
            image,
            batch_predict,
            top_labels=1,
            num_samples=1000,
            segmentation_fn=lambda img: slic(img, n_segments=50, compactness=10, sigma=1)
        )
        return explanation.image, explanation.segments

    def _process_attribution(self, attribution: np.ndarray) -> np.ndarray:
        """
        Process the attribution (averaging over channels, resizing, normalization, and smoothing).
        """
        # Average across channels if needed and squeeze extra dimensions
        attribution = np.mean(attribution, axis=1).squeeze()
        attribution = cv2.resize(attribution, (224, 224))
        attribution = (attribution - attribution.min()) / (attribution.max() - attribution.min() + 1e-8)
        return cv2.GaussianBlur(attribution, (5, 5), 0)

    def _create_overlay(self, image: np.ndarray, attribution: np.ndarray) -> np.ndarray:
        """Overlay a heatmap on the image."""
        heatmap = cv2.applyColorMap(np.uint8(attribution * 255), cv2.COLORMAP_JET)
        return cv2.addWeighted(image, 0.7, heatmap, 0.3, 0)

    def _save_visualization(self, fig, filename: str, pred_class: str):
        """Save the matplotlib figure to disk."""
        save_path = os.path.join(self.save_dir, pred_class, filename)
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, bbox_inches='tight', dpi=150)
        plt.close(fig)

    def analyze_image(self, image_path: str):
        """
        Process a single image: predict its class, compute explanations (DeepLIFT, SHAP, LIME,
        and attention) and save the visualizations.
        """
        try:
            # Load and preprocess the image for the model
            pil_img = Image.open(image_path).convert("RGB").resize((224, 224))
            image_tensor = self.transform(pil_img).unsqueeze(0).to(self.device)
            # Also get the original image (in pixel space) for overlaying / SHAP processing
            original_image = np.array(pil_img)

            # Get the model prediction
            pred_class_idx, confidence = self._get_prediction(image_tensor)
            pred_class = self.class_names[pred_class_idx]
            base_name = os.path.splitext(os.path.basename(image_path))[0]
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

            # -------------------------------
            # Generate Gradient-based Explanations
            # -------------------------------
            methods = {
                'DeepLIFT': self.generate_deeplift_attribution(image_tensor, target=pred_class_idx),
                'SHAP': self.generate_shap_attribution(image_tensor, target=pred_class_idx,
                                                       original_image=original_image),
            }

            for method_name, attribution in methods.items():
                fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
                ax1.imshow(original_image)
                ax1.set_title(f"Original\n{pred_class} ({confidence:.1%})")
                ax1.axis('off')

                # Overlay the heatmap on the original image
                overlay = self._create_overlay(original_image, attribution)
                ax2.imshow(overlay)
                ax2.set_title(f"{method_name} Explanation")
                ax2.axis('off')

                filename = f"{base_name}_{method_name}_{timestamp}.png"
                self._save_visualization(fig, filename, pred_class)

            # -------------------------------
            # LIME Explanation
            # -------------------------------
            lime_img, segments = self.generate_lime_explanation(image_path)
            fig, ax = plt.subplots(figsize=(8, 8))
            ax.imshow(mark_boundaries(lime_img, segments))
            ax.set_title("LIME Segmentation")
            ax.axis('off')
            self._save_visualization(fig, f"{base_name}_LIME_{timestamp}.png", pred_class)

            # -------------------------------
            # Attention Map Visualization
            # -------------------------------
            # Assumes the attention map is a 1D vector that can be reshaped into a 14x14 grid.
            attn_weights = self.model.get_attention_weights(image_tensor).reshape(14, 14)
            attn_map = cv2.resize(attn_weights, (224, 224), interpolation=cv2.INTER_CUBIC)
            fig, ax = plt.subplots(figsize=(8, 8))
            ax.imshow(original_image)
            ax.imshow(attn_map, cmap='viridis', alpha=0.5)
            ax.set_title("Attention Map")
            ax.axis('off')
            self._save_visualization(fig, f"{base_name}_Attention_{timestamp}.png", pred_class)

        except Exception as e:
            print(f"Error processing {image_path}: {str(e)}")


# =======================================
# 3. Helper Function to Process a Dataset
# =======================================
def analyze_dataset(analyzer: XAIAnalyzer, test_folder: str, max_per_class: int = 15):
    for class_name in os.listdir(test_folder):
        class_path = os.path.join(test_folder, class_name)
        if not os.path.isdir(class_path):
            continue

        print(f"Processing {class_name}...")
        images = [f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))][:max_per_class]

        for img_name in images:
            img_path = os.path.join(class_path, img_name)
            print(f"  Analyzing {img_name}")
            analyzer.analyze_image(img_path)


# =======================================
# 4. Main Execution
# =======================================
if __name__ == "__main__":
    warnings.filterwarnings("ignore", category=UserWarning)

    # Initialize the model and load your checkpoint
    model = EnhancedViT()
    model_path = '/content/drive/My Drive/FYP/VIT2_HQ2_20250207/final_model_vit_20250207.pth'
    model.load_state_dict(torch.load(model_path))

    # Initialize the analyzer
    analyzer = XAIAnalyzer(
        model=model,
        class_names=['Glaucous_Winged_Gull', 'Slaty_Backed_Gull'],
        save_dir='/content/drive/My Drive/FYP/XAI_Results'
    )

    # Process your dataset
    analyze_dataset(
        analyzer=analyzer,
        test_folder='/content/drive/My Drive/FYP/Dataset/HQ2/train',
        max_per_class=15
    )


<ipython-input-18-e9085072a6e5>:300: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))


Processing Slaty_Backed_Gull...
  Analyzing 0H5A6787.JPG
Error processing /content/drive/My Drive/FYP/Dataset/HQ2/train/Slaty_Backed_Gull/0H5A6787.JPG: mean() received an invalid combination of arguments - got (dtype=NoneType, out=NoneType, axis=int, ), but expected one of:
 * (*, torch.dtype dtype = None)
 * (tuple of ints dim, bool keepdim = False, *, torch.dtype dtype = None)
 * (tuple of names dim, bool keepdim = False, *, torch.dtype dtype = None)

  Analyzing 0H5A8628.JPG
Error processing /content/drive/My Drive/FYP/Dataset/HQ2/train/Slaty_Backed_Gull/0H5A8628.JPG: mean() received an invalid combination of arguments - got (dtype=NoneType, out=NoneType, axis=int, ), but expected one of:
 * (*, torch.dtype dtype = None)
 * (tuple of ints dim, bool keepdim = False, *, torch.dtype dtype = None)
 * (tuple of names dim, bool keepdim = False, *, torch.dtype dtype = None)

  Analyzing 0H5A7391.JPG
Error processing /content/drive/My Drive/FYP/Dataset/HQ2/train/Slaty_Backed_Gull/0H5A7391.J

KeyboardInterrupt: 